# 🧠 3-Model Arithmetic & Reasoning Comparison
### Large-Scale Evaluation on Shared Math Equations (1-4 & 8-11 Digits -> 5-7 Digit Gap Generalization)

This notebook sets up and trains three models on Google Cloud TPU v5e-8 (~1-Hour Run) or TPU v5e-1 (~8-Hour Run):
1. **Model 1**: Character Math Tokenizer (100% Math with `<THINK>` scratchpad)
2. **Model 2**: Subword BPE Tokenizer (100% Math with Natural Word-CoT)
3. **Model 3**: Subword BPE Tokenizer (50% Math + 50% Philosophy - Kant, Descartes, Spinoza)

> **Shared Dataset Guarantee**: All 3 models train on the **exact same 2,500,000 equations** in the **exact same order**.
> **TPU v5e-8 High Throughput**: With 8 cores in parallel (global batch size 512), 1 hour delivers ~1.05 Billion tokens!

## Step 1: Install Dependencies

In [ ]:
!pip install -q tiktoken
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
try:
    import torch_xla.core.xla_model as xm
    devices = xm.get_xla_supported_devices()
    print(f"TPU Detected ({len(devices)} cores):", devices)
except Exception as e:
    print("TPU / XLA: Not detected (using GPU/CPU):", e)

## Step 2: Verify / Pre-Generate Shared Math Dataset

In [ ]:
from src.dataset import get_or_generate_shared_math_pairs
a_tensor, b_tensor = get_or_generate_shared_math_pairs()
print(f"Verified {len(a_tensor):,} shared equation pairs cached at data/shared_math_pairs.pt!")

## Step 3: Train Model 1 (Character Tokenizer - 100% Math with <THINK>)
- **TPU v5e-8 (~1 Hour on 8 Cores)**: `!python training/train_model1_char.py --hours 1.0`
- **TPU v5e-1 (~8 Hours on 1 Core)**: `!python training/train_model1_char.py --hours 8.0 --cores 1`
- **Quick Test (1 Minute)**: `!python training/train_model1_char.py --hours 0.02`

In [ ]:
!python training/train_model1_char.py --hours 1.0

## Step 4: Train Model 2 (Subword BPE - 100% Math with Word-CoT)
- **TPU v5e-8 (~1 Hour on 8 Cores)**: `!python training/train_model2_bpe.py --hours 1.0`
- **TPU v5e-1 (~8 Hours on 1 Core)**: `!python training/train_model2_bpe.py --hours 8.0 --cores 1`

In [ ]:
!python training/train_model2_bpe.py --hours 1.0

## Step 5: Train Model 3 (Subword BPE - 50% Math + 50% Philosophy)
- **TPU v5e-8 (~1 Hour on 8 Cores)**: `!python training/train_model3_philo.py --hours 1.0`
- **TPU v5e-1 (~8 Hours on 1 Core)**: `!python training/train_model3_philo.py --hours 8.0 --cores 1`

In [ ]:
!python training/train_model3_philo.py --hours 1.0

## Step 6: Interactive Testing & Attention Inspection

In [ ]:
!python benchmarks/interact.py --model_path checkpoints/model_1/model_1_final_1h.pt --prompt "123456+654321="